# Watch Keypoint Prediction Pipeline Visualization

This notebook visualizes each step of the prediction pipeline:
1. **YOLO Detection**: Detect and crop watch face
2. **LoFTR Matching**: Find feature correspondences with template
3. **Homography**: Compute geometric transformation
4. **Keypoint Projection**: Map template keypoints to query image

In [ ]:
import sys
from pathlib import Path
import yaml

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D

# Add project root to path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 10)

## 1. Configuration

In [ ]:
# Paths
IMAGES_DIR = project_root / "downloaded_images"
CONFIG_PATH = project_root / "prediction_server" / "config.yaml"
TEMPLATES_DIR = project_root / "templates"

# Device (use MPS on Mac for GPU acceleration)
DEVICE = "mps"  # Options: "mps", "cpu", "cuda"

print(f"Images: {IMAGES_DIR}")
print(f"Templates: {TEMPLATES_DIR}")
print(f"Device: {DEVICE}")

## 2. Select Watch to Analyze

In [ ]:
# List available watches
watch_folders = sorted([d for d in IMAGES_DIR.iterdir() if d.is_dir()])
print(f"Found {len(watch_folders)} watches:\n")
for i, folder in enumerate(watch_folders[:20], 1):
    face_images = list(folder.glob("*face*.jpg"))
    print(f"  {i:2d}. {folder.name:<30s} ({len(face_images)} face images)")
if len(watch_folders) > 20:
    print(f"  ... and {len(watch_folders) - 20} more")

In [ ]:
# Select a watch model to analyze
WATCH_ID = "AP_roakpnd_054"  # Change this to analyze different watches

watch_folder = IMAGES_DIR / WATCH_ID
if not watch_folder.exists():
    raise FileNotFoundError(f"Watch folder not found: {watch_folder}")

# Get all face images for this watch
face_images = sorted(watch_folder.glob("*face*.jpg"))
print(f"\nSelected: {WATCH_ID}")
print(f"Found {len(face_images)} face images:\n")
for i, img in enumerate(face_images, 1):
    print(f"  {i:2d}. {img.name}")

In [ ]:
# Select specific image
IMAGE_NAME = face_images[0].name if face_images else None  # First image by default
IMAGE_PATH = IMAGES_DIR / WATCH_ID / IMAGE_NAME

if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

print(f"Selected image: {IMAGE_NAME}")

# Load image
img_bgr = cv2.imread(str(IMAGE_PATH))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_h, img_w = img_rgb.shape[:2]

print(f"Image size: {img_w} × {img_h}")

# Display
plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.title(f"Original Image: {IMAGE_NAME}\nSize: {img_w} × {img_h}", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Initialize Pipeline Components

In [ ]:
from prediction_server.pipelines.homography_keypoints import HomographyKeypointsPipeline
from prediction_server.pipelines.yolo_utils import YOLODetector
from prediction_server.pipelines.loftr_utils import LoFTRMatcher
from prediction_server.core.template_loader import TemplateLoader
from utils.model_mapper import get_template_from_filename

# Load config
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

pipeline_config = config['pipeline']
print("Pipeline configuration:")
print(f"  YOLO confidence: {pipeline_config['yolo']['conf_threshold']}")
print(f"  LoFTR match threshold: {pipeline_config['loftr']['match_threshold']}")
print(f"  Homography min inliers: {pipeline_config['homography']['min_inliers']}")

# Override device
pipeline_config['yolo']['device'] = DEVICE
pipeline_config['loftr']['device'] = DEVICE

# Determine template model from filename
template_model = get_template_from_filename(IMAGE_NAME)
print(f"\nTemplate model: {template_model}")

# Set template configuration
pipeline_config['template']['templates_dir'] = str(TEMPLATES_DIR)
pipeline_config['template']['model'] = template_model

# Initialize pipeline
pipeline = HomographyKeypointsPipeline(config=pipeline_config)

print("\n✓ Pipeline initialized")

## 4. Step 1: YOLO Detection

In [ ]:
from ultralytics import YOLO

# Load YOLO model
yolo_model_path = project_root / "prediction_server" / "models" / "yolo_watch_face_best.pt"
yolo = YOLO(str(yolo_model_path))

# Run detection
yolo_results = yolo.predict(
    source=str(IMAGE_PATH),
    conf=pipeline_config['yolo']['conf_threshold'],
    device=DEVICE,
    verbose=False
)

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.imshow(img_rgb)

if len(yolo_results) > 0 and yolo_results[0].obb is not None and len(yolo_results[0].obb) > 0:
    obbs = yolo_results[0].obb
    print(f"✓ YOLO detected {len(obbs)} watch face(s)\n")
    
    for i, obb in enumerate(obbs):
        corners = obb.xyxyxyxy[0].cpu().numpy()
        confidence = obb.conf[0].item()
        
        print(f"Detection {i+1}: confidence={confidence:.3f}")
        
        # Draw oriented bounding box
        polygon = patches.Polygon(
            corners, linewidth=3, edgecolor='lime', facecolor='none',
            label=f'YOLO Detection (conf={confidence:.2f})'
        )
        ax.add_patch(polygon)
        
        # Draw corners
        ax.scatter(corners[:, 0], corners[:, 1], c='lime', s=100, marker='o', zorder=10)
        
        # Draw center
        center = corners.mean(axis=0)
        ax.plot(center[0], center[1], 'r+', markersize=20, markeredgewidth=3)
else:
    print("✗ YOLO failed to detect watch face")
    ax.text(img_w/2, img_h/2, 'NO DETECTION', 
           color='red', fontsize=24, ha='center', va='center',
           weight='bold', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_title(f"Step 1: YOLO Watch Face Detection", fontsize=16, weight='bold')
if len(yolo_results) > 0 and yolo_results[0].obb is not None and len(yolo_results[0].obb) > 0:
    ax.legend(loc='upper right', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Step 2: Template Image

In [ ]:
# Get template data
template_data = pipeline.template_data
template_rgb = cv2.cvtColor(template_data.template_image, cv2.COLOR_BGR2RGB)
tmpl_h, tmpl_w = template_rgb.shape[:2]

print(f"Template: {template_data.model_name}")
print(f"Size: {template_data.image_size}")
print(f"\nKeypoints (normalized):")
for kp_name, (x, y) in template_data.keypoints_norm.items():
    print(f"  {kp_name:>6s}: ({x:.4f}, {y:.4f})")

# Visualize template with keypoints
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.imshow(template_rgb)

keypoint_colors = {
    'top': 'red',
    'bottom': 'blue',
    'left': 'yellow',
    'right': 'purple',
    'center': 'orange'
}

for kp_name, (x_norm, y_norm) in template_data.keypoints_norm.items():
    x_px = x_norm * tmpl_w
    y_px = y_norm * tmpl_h
    color = keypoint_colors.get(kp_name, 'white')
    
    ax.plot(x_px, y_px, 'o', color=color, markersize=15,
            markeredgewidth=2, markeredgecolor='black', label=kp_name.capitalize())

ax.set_title(f"Template: {template_data.model_name}\n{tmpl_w}×{tmpl_h}px", 
             fontsize=16, weight='bold')
ax.legend(loc='upper right', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Step 3: LoFTR Feature Matching

In [ ]:
# Run YOLO detection and get cropped image (phase1)
phase1_img, num_det, yolo_conf, reason, obb_data = pipeline.yolo_detector.detect_and_align(
    img_bgr,
    padding_factor=pipeline.padding_factor,
    template_size=(template_data.image_size[1], template_data.image_size[0])
)

if phase1_img is None:
    print(f"✗ YOLO processing failed: {reason}")
else:
    print(f"✓ YOLO cropped image: {phase1_img.shape[1]}×{phase1_img.shape[0]}")
    
    # Run LoFTR matching
    print("\nRunning LoFTR matching...")
    mkpts0, mkpts1, mconf = pipeline.loftr_matcher.find_correspondences(
        phase1_img,
        template_data.template_image,
        match_threshold=pipeline.match_threshold
    )
    
    print(f"✓ Found {len(mkpts0)} feature matches")
    print(f"  Mean confidence: {mconf.mean():.3f}")
    print(f"  Median confidence: {np.median(mconf):.3f}")
    
    # Visualize matches
    phase1_rgb = cv2.cvtColor(phase1_img, cv2.COLOR_BGR2RGB)
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # Query image (cropped)
    axes[0].imshow(phase1_rgb)
    axes[0].scatter(mkpts0[:, 0], mkpts0[:, 1], c=mconf, s=50, cmap='viridis', alpha=0.7)
    axes[0].set_title(f"Query (Cropped)\n{len(mkpts0)} matches", fontsize=14, weight='bold')
    axes[0].axis('off')
    
    # Template image
    axes[1].imshow(template_rgb)
    axes[1].scatter(mkpts1[:, 0], mkpts1[:, 1], c=mconf, s=50, cmap='viridis', alpha=0.7)
    axes[1].set_title(f"Template\n{len(mkpts1)} matches", fontsize=14, weight='bold')
    axes[1].axis('off')
    
    plt.suptitle("Step 2: LoFTR Feature Matching", fontsize=16, weight='bold')
    plt.tight_layout()
    plt.show()
    
    # Store for next step
    matched_points = (mkpts0, mkpts1, mconf)

## 7. Step 4: Homography Estimation

In [ ]:
if phase1_img is not None and len(mkpts0) > 0:
    # Compute homography using RANSAC
    H, mask = cv2.findHomography(
        mkpts1, mkpts0,  # template -> query
        cv2.RANSAC,
        ransacReprojThreshold=pipeline_config['homography']['ransac_threshold']
    )
    
    if H is not None:
        inliers = mask.sum()
        print(f"✓ Homography computed")
        print(f"  Inliers: {inliers} / {len(mkpts0)} ({inliers/len(mkpts0)*100:.1f}%)")
        
        # Visualize inliers vs outliers
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        
        # Query image
        axes[0].imshow(phase1_rgb)
        axes[0].scatter(mkpts0[mask.ravel() == 1, 0], mkpts0[mask.ravel() == 1, 1],
                       c='lime', s=80, marker='o', label=f'Inliers ({inliers})', alpha=0.7)
        axes[0].scatter(mkpts0[mask.ravel() == 0, 0], mkpts0[mask.ravel() == 0, 1],
                       c='red', s=40, marker='x', label=f'Outliers ({len(mkpts0)-inliers})', alpha=0.5)
        axes[0].set_title("Query (Cropped)", fontsize=14, weight='bold')
        axes[0].legend(loc='upper right', fontsize=12)
        axes[0].axis('off')
        
        # Template image
        axes[1].imshow(template_rgb)
        axes[1].scatter(mkpts1[mask.ravel() == 1, 0], mkpts1[mask.ravel() == 1, 1],
                       c='lime', s=80, marker='o', label=f'Inliers ({inliers})', alpha=0.7)
        axes[1].scatter(mkpts1[mask.ravel() == 0, 0], mkpts1[mask.ravel() == 0, 1],
                       c='red', s=40, marker='x', label=f'Outliers ({len(mkpts1)-inliers})', alpha=0.5)
        axes[1].set_title("Template", fontsize=14, weight='bold')
        axes[1].legend(loc='upper right', fontsize=12)
        axes[1].axis('off')
        
        plt.suptitle("Step 3: Homography RANSAC", fontsize=16, weight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print("✗ Homography computation failed")

## 7a. Visualize Homography Transformation

## 7b. Diagnostic: Check Rotation Inverse Accuracy

In [ ]:
if phase1_img is not None and 'H' in locals() and H is not None:
    # The homography H maps template -> query, so we need the inverse to warp query -> template
    H_inv = np.linalg.inv(H)
    
    # Warp query to template space using inverse homography
    phase1_h, phase1_w = phase1_img.shape[:2]
    warped_query = cv2.warpPerspective(phase1_rgb, H_inv, (tmpl_w, tmpl_h))
    
    # Create overlay: template with alpha blending
    alpha = 0.5  # Transparency for overlay
    overlay = cv2.addWeighted(template_rgb, alpha, warped_query, 1-alpha, 0)
    
    # Transform template keypoints to pixel coordinates
    template_kpts_px = {}
    for kp_name, (x_norm, y_norm) in template_data.keypoints_norm.items():
        template_kpts_px[kp_name] = (x_norm * tmpl_w, y_norm * tmpl_h)
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(20, 20))
    
    # 1. Warped query image alone
    axes[0, 0].imshow(warped_query)
    axes[0, 0].set_title("Warped Query (transformed to template space)", fontsize=14, weight='bold')
    axes[0, 0].axis('off')
    
    # 2. Template image alone
    axes[0, 1].imshow(template_rgb)
    axes[0, 1].set_title("Template Reference", fontsize=14, weight='bold')
    axes[0, 1].axis('off')
    
    # 3. Overlay of warped query on template
    axes[1, 0].imshow(overlay)
    axes[1, 0].set_title(f"Overlay (alpha={alpha})\nShows alignment quality", fontsize=14, weight='bold')
    axes[1, 0].axis('off')
    
    # 4. Overlay with keypoints
    axes[1, 1].imshow(overlay)
    for kp_name, (x_px, y_px) in template_kpts_px.items():
        color = keypoint_colors.get(kp_name, 'white')
        axes[1, 1].plot(x_px, y_px, 'o', color=color, markersize=15,
                       markeredgewidth=3, markeredgecolor='black', 
                       label=kp_name.capitalize())
        # Add crosshair
        axes[1, 1].plot([x_px-20, x_px+20], [y_px, y_px], color=color, linewidth=2, alpha=0.7)
        axes[1, 1].plot([x_px, x_px], [y_px-20, y_px+20], color=color, linewidth=2, alpha=0.7)
    
    axes[1, 1].set_title("Overlay with Template Keypoints", fontsize=14, weight='bold')
    axes[1, 1].legend(loc='upper right', fontsize=11)
    axes[1, 1].axis('off')
    
    plt.suptitle("Step 3a: Homography Transformation Visualization", fontsize=16, weight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Homography transformation applied")
    print(f"  Template size: {tmpl_w}×{tmpl_h}")
    print(f"  Alignment quality: {inliers}/{len(mkpts0)} inliers ({inliers/len(mkpts0)*100:.1f}%)")
    print(f"\nThe warped query should now align with the template.")
    print(f"Template keypoints (colored dots) will be inverse-transformed")
    print(f"back to the original query image to get the final predictions.")
else:
    print("⚠️  Homography not available - skipping transformation visualization")

In [ ]:
if phase1_img is not None and obb_data and obb_data.get('transform_params'):
    transform_params = obb_data['transform_params']
    
    if transform_params['type'] == 'crop_rotate_resize':
        print("="*80)
        print("ROTATION INVERSION DIAGNOSTIC")
        print("="*80)
        
        # Get the rotation matrix used by YOLO
        rotation_matrix = np.array(transform_params['rotation_matrix'])  # 2x3 matrix
        rotation_deg = transform_params['rotation_deg']
        crop_center = transform_params['crop_center']
        
        print(f"\nYOLO Rotation:")
        print(f"  Rotation angle: {rotation_deg:.4f} degrees")
        print(f"  Crop center: ({crop_center[0]:.2f}, {crop_center[1]:.2f})")
        print(f"\n  Rotation Matrix (from cv2.getRotationMatrix2D):")
        print(f"    {rotation_matrix}")
        
        # Test point: use crop center offset by (100, 0) to the right
        test_pt = np.array([crop_center[0] + 100, crop_center[1], 1.0])
        
        # Forward transformation using cv2 matrix
        rotated_pt_cv2 = rotation_matrix @ test_pt
        print(f"\nTest Point: ({test_pt[0]:.2f}, {test_pt[1]:.2f})")
        print(f"After forward rotation (cv2): ({rotated_pt_cv2[0]:.2f}, {rotated_pt_cv2[1]:.2f})")
        
        # Manual inverse rotation (what the pipeline does)
        x_rotated, y_rotated = rotated_pt_cv2[0], rotated_pt_cv2[1]
        x_centered = x_rotated - crop_center[0]
        y_centered = y_rotated - crop_center[1]
        
        angle_rad = np.radians(rotation_deg)  # Positive angle for inverse
        cos_a = np.cos(angle_rad)
        sin_a = np.sin(angle_rad)
        x_unrotated_manual = cos_a * x_centered - sin_a * y_centered + crop_center[0]
        y_unrotated_manual = sin_a * x_centered + cos_a * y_centered + crop_center[1]
        
        print(f"After manual inverse rotation: ({x_unrotated_manual:.2f}, {y_unrotated_manual:.2f})")
        
        # Correct inverse using cv2.invertAffineTransform
        rotation_matrix_inv_cv2 = cv2.invertAffineTransform(rotation_matrix)
        pt_inv_cv2 = rotation_matrix_inv_cv2 @ np.array([rotated_pt_cv2[0], rotated_pt_cv2[1], 1.0])
        
        print(f"After cv2 inverse rotation: ({pt_inv_cv2[0]:.2f}, {pt_inv_cv2[1]:.2f})")
        print(f"Original point: ({test_pt[0]:.2f}, {test_pt[1]:.2f})")
        
        # Calculate errors
        manual_error = np.sqrt((x_unrotated_manual - test_pt[0])**2 + (y_unrotated_manual - test_pt[1])**2)
        cv2_error = np.sqrt((pt_inv_cv2[0] - test_pt[0])**2 + (pt_inv_cv2[1] - test_pt[1])**2)
        
        print(f"\n" + "="*80)
        print("ERROR ANALYSIS:")
        print(f"  Manual inverse error: {manual_error:.4f} pixels")
        print(f"  cv2.invertAffineTransform error: {cv2_error:.4f} pixels")
        print(f"  Difference: {abs(manual_error - cv2_error):.4f} pixels")
        
        if manual_error > 0.1:
            print(f"\n⚠️  FOUND THE BUG!")
            print(f"  The manual rotation inverse has {manual_error:.2f} pixels of error.")
            print(f"  This explains the ~1 degree rotation offset you're seeing.")
            print(f"\n  Root cause: The code uses manual rotation math instead of")
            print(f"  cv2.invertAffineTransform(), which properly inverts the")
            print(f"  rotation matrix including all numerical precision.")
        else:
            print(f"\n✓ Manual rotation appears accurate (error < 0.1 pixels)")
        
        print("="*80)
    else:
        print("⚠️  Transform type is 'resize_only', no rotation to diagnose")
else:
    print("⚠️  No transform params available")

## 8. Step 5: Run Complete Pipeline & Visualize Results

In [ ]:
# Run full pipeline
print("Running complete pipeline...")
result = pipeline.predict(IMAGE_PATH)  # Pass Path object, not string

print(f"\nResult:")
print(f"  Success: {result.success}")
print(f"  Confidence: {result.confidence:.3f}")
print(f"  Method: {result.debug_info.get('method', 'Unknown') if result.debug_info else 'Unknown'}")

if result.debug_info:
    print(f"\nDebug info:")
    for key, value in result.debug_info.items():
        print(f"  {key}: {value}")

if result.error_message:
    print(f"\nError: {result.error_message}")

In [ ]:
# Visualize final predictions
if result.keypoints:
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    ax.imshow(img_rgb)
    
    keypoint_colors = {
        'top': 'red',
        'bottom': 'blue',
        'left': 'yellow',
        'right': 'purple',
        'center': 'orange'
    }
    
    print("\nPredicted keypoints (pixel coordinates):")
    for kp_name, color in keypoint_colors.items():
        coords_norm = getattr(result.keypoints, kp_name)
        x_px = coords_norm[0] * img_w
        y_px = coords_norm[1] * img_h
        
        print(f"  {kp_name:>6s}: ({x_px:7.1f}, {y_px:7.1f}) px")
        
        # Draw keypoint
        ax.plot(x_px, y_px, 'o', color=color, markersize=15,
               markeredgewidth=2, markeredgecolor='black', label=kp_name.capitalize())
        
        # Add label
        ax.text(x_px + 30, y_px - 30, kp_name.capitalize(),
               color=color, fontsize=12, weight='bold',
               bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
    
    method = result.debug_info.get('method', 'Unknown') if result.debug_info else 'Unknown'
    ax.set_title(f"Step 4: Final Predictions\nMethod: {method} | Confidence: {result.confidence:.3f}",
                 fontsize=16, weight='bold')
    ax.legend(loc='upper right', fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("✗ No keypoints predicted")

## 9. Batch Visualize All Images for This Watch

In [ ]:
# Predict on all face images for this watch
print(f"Processing {len(face_images)} images for {WATCH_ID}...\n")

results_summary = []

for img_path in face_images:
    result = pipeline.predict(img_path)  # Pass Path object, not string
    method = result.debug_info.get('method', 'Unknown') if result.debug_info else 'Unknown'
    
    results_summary.append({
        'image': img_path.name,
        'success': result.success,
        'confidence': result.confidence,
        'method': method
    })
    
    status = '✓' if result.success else '✗'
    print(f"{status} {img_path.name:<40s} | conf={result.confidence:.3f} | method={method}")

# Summary statistics
print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
successful = sum(1 for r in results_summary if r['success'])
print(f"Success rate: {successful}/{len(results_summary)} ({successful/len(results_summary)*100:.1f}%)")
print(f"\nMethod breakdown:")
from collections import Counter
methods = Counter(r['method'] for r in results_summary)
for method, count in methods.most_common():
    print(f"  {method}: {count} ({count/len(results_summary)*100:.1f}%)")

In [ ]:
# Visualize grid of predictions
n_images = min(len(face_images), 9)  # Show up to 9 images
cols = 3
rows = (n_images + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, 6*rows))
axes = axes.flatten() if n_images > 1 else [axes]

for idx, img_path in enumerate(face_images[:n_images]):
    # Load image
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    # Predict - pass Path object, not string
    result = pipeline.predict(img_path)
    
    # Display
    axes[idx].imshow(img_rgb)
    
    if result.keypoints:
        for kp_name, color in keypoint_colors.items():
            coords_norm = getattr(result.keypoints, kp_name)
            x_px = coords_norm[0] * w
            y_px = coords_norm[1] * h
            axes[idx].plot(x_px, y_px, 'o', color=color, markersize=10,
                          markeredgewidth=2, markeredgecolor='black')
    
    method = result.debug_info.get('method', 'Unknown') if result.debug_info else 'Unknown'
    status = '✓' if result.success else '✗'
    axes[idx].set_title(f"{status} {img_path.name}\n{method} | conf={result.confidence:.2f}",
                        fontsize=10)
    axes[idx].axis('off')

# Hide unused subplots
for idx in range(n_images, len(axes)):
    axes[idx].axis('off')

plt.suptitle(f"Batch Predictions: {WATCH_ID}", fontsize=16, weight='bold')
plt.tight_layout()
plt.show()